# 1. Library calling

In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import datetime
import warnings
import numpy as np
from IPython.display import clear_output
# Ignore all warnings (not recommended in general)
warnings.filterwarnings("ignore")
from selenium.webdriver.support.ui import Select
from janitor import xlsx_table

# 2. Defining the Product Information and Location

In [209]:
df=pd.read_excel(r"C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\14_Ignition_Coil\ACES & PIES_Ignition Coils 2024-01-24.xlsm",sheet_name='ACES')
#df=xlsx_table(IFolder+'\\'+filename,sheetname="Photos", table="Data_List")
df.head()

,PartNumber,Brand,AppID,GroupID,Qty,MfrLabel,DisplayOrder,PartType,Make,Model,...,QualifierText,AssetName,AssetType,AssetRepresentation,AssetItemRef,AssetItemOrder,LanguageCode,LanguageName,ParentPartNumber,ParentBrand
0,53001,Autolite,NaN,NaN,1,NaN,NaN,Ignition Coil,Cadillac,Escalade,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,53001,Autolite,NaN,NaN,1,NaN,NaN,Ignition Coil,Cadillac,Escalade,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,53001,Autolite,NaN,NaN,1,NaN,NaN,Ignition Coil,Cadillac,Escalade ESV,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,53001,Autolite,NaN,NaN,1,NaN,NaN,Ignition Coil,Cadillac,Escalade EXT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,53001,Autolite,NaN,NaN,1,NaN,NaN,Ignition Coil,Chevrolet,Avalanche 1500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [261]:
df1=df[['Make','Model','EngineBaseLiter','EngineBaseBlockType','EngineBaseCylinder','YearFrom','YearTo']]
df1.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1458 entries, 0 to 1457
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Make                 1458 non-null   object 
 1   Model                1458 non-null   object 
 2   EngineBaseLiter      1458 non-null   float64
 3   EngineBaseBlockType  1458 non-null   object 
 4   EngineBaseCylinder   1458 non-null   int64  
 5   YearFrom             1458 non-null   int64  
 6   YearTo               1458 non-null   int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 79.9+ KB


In [262]:
df1['Years'] = df1.apply(lambda row: list(range(row['YearFrom'], row['YearTo'] + 1)), axis=1)
df1.drop(columns={'YearTo','YearFrom'})
df1=df1.explode('Years',ignore_index=True)
df1=df1.drop(columns={'YearTo','YearFrom'})


In [263]:
#selectedengine='5.3_8_V'

df1['engcomb1']=df1['EngineBaseCylinder'].astype(str)+"_"+df1['EngineBaseLiter'].astype(str)+"_"+df1['EngineBaseBlockType']
df1['engcomb2']=df1['EngineBaseLiter'].astype(str)+"_"+df1['EngineBaseCylinder'].astype(str)+"_"+df1['EngineBaseBlockType']

In [213]:
df1

,Make,Model,EngineBaseLiter,EngineBaseBlockType,EngineBaseCylinder,Years,engcomb
0,Cadillac,Escalade,5.3,V,8,2002,5.3_8_V
1,Cadillac,Escalade,5.3,V,8,2003,5.3_8_V
2,Cadillac,Escalade,5.3,V,8,2004,5.3_8_V
3,Cadillac,Escalade,5.3,V,8,2005,5.3_8_V
4,Cadillac,Escalade,6.0,V,8,2002,6.0_8_V
...,...,...,...,...,...,...,...
6916,Subaru,Outback,3.0,H,6,2005,3.0_6_H
6917,Subaru,Outback,3.0,H,6,2006,3.0_6_H
6918,Subaru,Outback,3.0,H,6,2007,3.0_6_H
6919,Subaru,Outback,3.0,H,6,2008,3.0_6_H


In [264]:
df1

,Make,Model,EngineBaseLiter,EngineBaseBlockType,EngineBaseCylinder,Years,engcomb1,engcomb2
0,Cadillac,Escalade,5.3,V,8,2002,8_5.3_V,5.3_8_V
1,Cadillac,Escalade,5.3,V,8,2003,8_5.3_V,5.3_8_V
2,Cadillac,Escalade,5.3,V,8,2004,8_5.3_V,5.3_8_V
3,Cadillac,Escalade,5.3,V,8,2005,8_5.3_V,5.3_8_V
4,Cadillac,Escalade,6.0,V,8,2002,8_6.0_V,6.0_8_V
...,...,...,...,...,...,...,...,...
6916,Subaru,Outback,3.0,H,6,2005,6_3.0_H,3.0_6_H
6917,Subaru,Outback,3.0,H,6,2006,6_3.0_H,3.0_6_H
6918,Subaru,Outback,3.0,H,6,2007,6_3.0_H,3.0_6_H
6919,Subaru,Outback,3.0,H,6,2008,6_3.0_H,3.0_6_H


In [3]:
with pd.ExcelWriter(r"C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\14_Ignition_Coil"+'\\'+'NGK_Export_Ignition_Coil_Input_2'+'.xlsx') as writer:  # doctest: +SKIP
    df1.to_excel(writer,index=True, sheet_name='Raw')

NameError: name 'df1' is not defined

# Start From here 

In [7]:
df1=pd.read_excel(r"C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\14_Ignition_Coil\NGK_Export_Ignition_Coil_Input_2.xlsx",sheet_name='Raw')

In [8]:
df1.head()

,Make,Model,Years,EngineBaseLiter,EngineBaseBlockType,EngineBaseCylinder,engcomb1,engcomb2
0,Ram,1500,2013,4.7,V,8,8_4.7_V,4.7_8_V
1,Ram,1500,2011,3.7,V,6,6_3.7_V,3.7_6_V
2,Ram,1500,2012,3.7,V,6,6_3.7_V,3.7_6_V
3,Ram,1500,2011,4.7,V,8,8_4.7_V,4.7_8_V
4,Ram,1500,2012,4.7,V,8,8_4.7_V,4.7_8_V


In [81]:
path= 'C://chromedriver.exe'
driver=webdriver.Chrome()
wait=WebDriverWait(driver, 10)
driver.get('https://ngksparkplugs.com/en/part-finder')

In [82]:
driver.find_element(By.CSS_SELECTOR, '[data-tracking-value="Cars, Trucks, SUV"]').click()

In [83]:
frame_0 = driver.find_element(By.CLASS_NAME,'ngk-embedded-content')
driver.switch_to.frame(frame_0)


In [17]:
cols2 =['Sl.No'] #,'Review_Mentions'
df2= pd.DataFrame(columns=cols2)
count=0

In [18]:
from tqdm import tqdm

In [88]:
for i in tqdm(range(1799,1805)):
    selectedyear=df1.loc[i,'Years']
    selectedmake=df1.loc[i,'Make']
    selectedmodel=df1.loc[i,'Model']
    selectedengine1=df1.loc[i,'engcomb1']
    selectedengine2=df1.loc[i,'engcomb2']
    print(i,"|",selectedyear," ",selectedmake," ",selectedmodel," ",selectedengine1,selectedengine2)
    print(len(df2))
    year=driver.find_element(By.CSS_SELECTOR,'[aria-labelledby="year-select"]')
    year.click()
    y1=df1.loc[i,'Years']-1
    sleep(1)
    driver.find_element(By.CSS_SELECTOR,f'[data-value="{y1}"]').click()
    year=driver.find_element(By.CSS_SELECTOR,'[aria-labelledby="year-select"]')
    sleep(1)
    year.click()
    sy=driver.find_element(By.CSS_SELECTOR,f'[data-value="{selectedyear}"]')
    sleep(2.5)
    sy.click()
    sleep(1)
    #-----------------------------------------------------------------------------------------------------------------------------------------
    make=driver.find_element(By.CSS_SELECTOR,'[aria-labelledby="make-select"]')
    sleep(1)
    make.click()

    sm=driver.find_element(By.CSS_SELECTOR,f'[data-value="{selectedmake}"]')
    sleep(2.5)
    sm.click()
    #-----------------------------------------------------------------------------------------------------------------------------------------
    model=driver.find_element(By.CSS_SELECTOR,'[aria-labelledby="model-select"]')
    sleep(1)
    model.click()

    smo=driver.find_element(By.CSS_SELECTOR,f'[data-value="{selectedmodel}"]')
    sleep(3)
    smo.click()
    #-----------------------------------------------------------------------------------------------------------------------------------------
    engine=driver.find_element(By.CSS_SELECTOR,'[aria-labelledby="engine-select"]')
    sleep(1)
    engine.click()
    ptype=driver.find_element(By.CSS_SELECTOR,'[aria-labelledby="product-type-select"]')
    try:
        try:
            se=driver.find_element(By.CSS_SELECTOR,f'[data-value="{selectedengine1}"]')
            sleep(2.5)
            se.click()
        except:
            se=driver.find_element(By.CSS_SELECTOR,f'[data-value="{selectedengine2}"]')
            sleep(2.5)
            se.click()
        sleep(1)
        #-----------------------------------------------------------------------------------------------------------------------------------------
        ptype.click()
        sleep(1)
        # clear=driver.find_element(By.XPATH,f"//li[text()='All Products']")
        # clear.click()
        #ptype.clear()
        try:
            ProductType='NGK Ignition Coils, Wire Sets & COP Boots'
            sp=driver.find_element(By.XPATH,f"//li[text()='{ProductType}']")
            sp.click()
            #-----------------------------------------------------------------------------------------------------------------------------------------
            sleep(2)
            lbs=driver.find_elements(By.CLASS_NAME,'css-8e4lkk')
            vls=driver.find_elements(By.CLASS_NAME,'css-5kqnit')
            boxn=driver.find_elements(By.CLASS_NAME,'css-aoeo82')
            for b in (boxn):
                lbs=b.find_elements(By.CLASS_NAME,'css-8e4lkk')
                vls=b.find_elements(By.CLASS_NAME,'css-5kqnit')
                for lb,vl in zip(lbs,vls):
                    df2.loc[count,'Sl.No']=i
                    df2.loc[count,'Years']=df1.loc[i,'Years']
                    df2.loc[count,'Make']=df1.loc[i,'Make']
                    df2.loc[count,'Model']=df1.loc[i,'Model']
                    df2.loc[count,'EngineBaseLiter']=df1.loc[i,'EngineBaseLiter']
                    df2.loc[count,'EngineBaseCylinder']=df1.loc[i,'EngineBaseCylinder']
                    df2.loc[count,'EngineBaseBlockType']=df1.loc[i,'EngineBaseBlockType']
                    df2.loc[count,lb.text]=vl.text
                count=count+1
        except:
            pass
        ptype.click()
        clear=driver.find_element(By.XPATH,f"//li[text()='All Products']")
        clear.click()
    except:
        dummy = driver.find_element(By.CLASS_NAME, 'css-1km1ehz')
        dummy.click()
        df2.loc[count,'Sl.No']=i
        df2.loc[count,'Years']=df1.loc[i,'Years']
        df2.loc[count,'Make']=df1.loc[i,'Make']
        df2.loc[count,'Model']=df1.loc[i,'Model']
        df2.loc[count,'EngineBaseLiter']=df1.loc[i,'EngineBaseLiter']
        df2.loc[count,'EngineBaseCylinder']=df1.loc[i,'EngineBaseCylinder']
        df2.loc[count,'EngineBaseBlockType']=df1.loc[i,'EngineBaseBlockType']
        df2.loc[count,'Part #']="No combination available in NGK"
        df2.loc[count,'Fitment Notes']='Unsuccessful extraction'
        count=count+1
    sleep(1)    
    clear_output(wait=True)

 33%|███▎      | 2/6 [00:45<01:31, 22.77s/it]

1801 | 2023   Lexus   RC350   6_3.5_V 3.5_6_V
2189


 33%|███▎      | 2/6 [00:56<01:52, 28.23s/it]


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":"[data-value="RC350"]"}
  (Session info: chrome=122.0.6261.112); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x00007FF67FF5AD02+56930]
	(No symbol) [0x00007FF67FECF602]
	(No symbol) [0x00007FF67FD842E5]
	(No symbol) [0x00007FF67FDC98ED]
	(No symbol) [0x00007FF67FDC9A2C]
	(No symbol) [0x00007FF67FE0A967]
	(No symbol) [0x00007FF67FDEBCDF]
	(No symbol) [0x00007FF67FE081E2]
	(No symbol) [0x00007FF67FDEBA43]
	(No symbol) [0x00007FF67FDBD438]
	(No symbol) [0x00007FF67FDBE4D1]
	GetHandleVerifier [0x00007FF6802D6F8D+3711213]
	GetHandleVerifier [0x00007FF6803304CD+4077101]
	GetHandleVerifier [0x00007FF68032865F+4044735]
	GetHandleVerifier [0x00007FF67FFF9736+706710]
	(No symbol) [0x00007FF67FEDB8DF]
	(No symbol) [0x00007FF67FED6AC4]
	(No symbol) [0x00007

In [89]:
print(len(df2))
df2.tail()
#df2.head()
df2

2189


,Sl.No,Years,Make,Model,EngineBaseLiter,EngineBaseCylinder,EngineBaseBlockType,Product Type,Part #,Stock #,Fitment Notes,Qty
0,2000,2011.0,Chevrolet,Silverado 1500,6.0,8.0,V,NGK Coil Near Plug Ignition Coil,U5293,48933,with Round Ignition Coil,8
1,2000,2011.0,Chevrolet,Silverado 1500,6.0,8.0,V,NGK Coil Near Plug Ignition Coil,U5132,48713,with Square Type Coil,8
2,2000,2011.0,Chevrolet,Silverado 1500,6.0,8.0,V,NGK MOD High-Performance Ignition Coil Multi-Pack,M5293-8,49471,with Round Ignition Coil,1
3,2000,2011.0,Chevrolet,Silverado 1500,6.0,8.0,V,NGK MOD High-Performance Ignition Coil Multi-Pack,M5132-8,49472,with Square Type Coil,1
4,2000,2011.0,Chevrolet,Silverado 1500,6.0,8.0,V,NGK Spark Plug Wire Set,RC-GMX113,51440,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...
2184,1797,2021.0,Lexus,RC350,3.5,6.0,V,NGK COP (Pencil Type) Ignition Coil,U5403,49186,NaN,6
2185,1798,2021.0,Lexus,RC350,3.5,6.0,V,NGK COP (Pencil Type) Ignition Coil,U5403,49186,NaN,6
2186,1799,2022.0,Lexus,RC350,3.5,6.0,V,NGK COP (Pencil Type) Ignition Coil,U5403,49186,NaN,6
2187,1799,2022.0,Lexus,RC350,3.5,6.0,V,NGK COP (Pencil Type) Ignition Coil,U5403,49186,NaN,6


# 4. Defining the Dataframe and Extracting the data into the Dataframe

In [90]:
with pd.ExcelWriter(r"C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\14_Ignition_Coil"+'\\'+'NGK_Export_Ignition_Coil_P2_1000-1460-1800-2000-2531'+'.xlsx') as writer:  # doctest: +SKIP
    df2.to_excel(writer,index=True, sheet_name='Raw')

# 99. Archived Codes

In [515]:
cols2 =['Sl.No'] #,'Review_Mentions'
df3= pd.DataFrame(columns=cols2)
count=0

In [ ]:
#New Content
for i in tqdm(range(1966,len(df1))):
    selectedyear=df1.loc[i,'Years']
    selectedmake=df1.loc[i,'Make']
    selectedmodel=df1.loc[i,'Model']
    selectedengine1=df1.loc[i,'engcomb1']
    selectedengine2=df1.loc[i,'engcomb2']
    print(i,"|",selectedyear," ",selectedmake," ",selectedmodel," ",selectedengine1,selectedengine2)
    print(len(df2))
    year=wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,'[aria-labelledby="year-select"]')))
    year.click()
    y1=df1.loc[i,'Years']+1
    sleep(1)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,f'[data-value="{y1}"]'))).click()
    year=wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,'[aria-labelledby="year-select"]')))    
    year.click()
    sy=wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,f'[data-value="{selectedyear}"]')))    
    sy.click()
    sleep(1)    
    #-----------------------------------------------------------------------------------------------------------------------------------------
    make=wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,'[aria-labelledby="make-select"]')))    
    make.click()
    sm=wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,f'[data-value="{selectedmake}"]')))    
    sm.click()
    sleep(1)
    #-----------------------------------------------------------------------------------------------------------------------------------------
    model=wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,'[aria-labelledby="model-select"]')))    
    model.click()
    smo=wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,f'[data-value="{selectedmodel}"]')))    
    smo.click()
    sleep(1)
    #-----------------------------------------------------------------------------------------------------------------------------------------
    engine=wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,'[aria-labelledby="engine-select"]')))    
    engine.click()
    ptype=wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,'[aria-labelledby="product-type-select"]')))
    try:
        try:
            se=wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,f'[data-value="{selectedengine2}"]')))            
            se.click()
        except:
            se=wait.until(EC.presence_of_element_located((By.CSS_SELECTOR,f'[data-value="{selectedengine1}"]')))            
            se.click()
        
        #-----------------------------------------------------------------------------------------------------------------------------------------
        ptype.click()        
        # clear=wait.until(EC.presence_of_element_located((By.XPATH,f"//li[text()='All Products']")
        # clear.click()
        #ptype.clear()
        try:
            ProductType='NGK Ignition Coils, Wire Sets & COP Boots'
            sp=wait.until(EC.presence_of_element_located((By.XPATH,f"//li[text()='{ProductType}']")))
            sp.click()
            #-----------------------------------------------------------------------------------------------------------------------------------------
            sleep(3)
            lbs=wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME,'css-8e4lkk')))
            vls=wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME,'css-5kqnit')))
            boxn=wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME,'css-aoeo82')))
            for b in (boxn):
                lbs=b.find_elements(By.CLASS_NAME,'css-8e4lkk')
                vls=b.find_elements(By.CLASS_NAME,'css-5kqnit')
                for lb,vl in zip(lbs,vls):
                    df2.loc[count,'Sl.No']=i
                    df2.loc[count,'Years']=df1.loc[i,'Years']
                    df2.loc[count,'Make']=df1.loc[i,'Make']
                    df2.loc[count,'Model']=df1.loc[i,'Model']
                    df2.loc[count,'EngineBaseLiter']=df1.loc[i,'EngineBaseLiter']
                    df2.loc[count,'EngineBaseCylinder']=df1.loc[i,'EngineBaseCylinder']
                    df2.loc[count,'EngineBaseBlockType']=df1.loc[i,'EngineBaseBlockType']
                    df2.loc[count,lb.text]=vl.text
                count=count+1
        except:
            pass
        ptype.click()
        clear=wait.until(EC.presence_of_element_located((By.XPATH,f"//li[text()='All Products']")))
        clear.click()
    except:
        dummy = wait.until(EC.presence_of_element_located((By.CLASS_NAME, 'css-1km1ehz')))
        dummy.click()
        df2.loc[count,'Sl.No']=i
        df2.loc[count,'Years']=df1.loc[i,'Years']
        df2.loc[count,'Make']=df1.loc[i,'Make']
        df2.loc[count,'Model']=df1.loc[i,'Model']
        df2.loc[count,'EngineBaseLiter']=df1.loc[i,'EngineBaseLiter']
        df2.loc[count,'EngineBaseCylinder']=df1.loc[i,'EngineBaseCylinder']
        df2.loc[count,'EngineBaseBlockType']=df1.loc[i,'EngineBaseBlockType']
        df2.loc[count,'Part #']="No combination available in NGK"
        df2.loc[count,'Fitment Notes']='Unsuccessful extraction'
        count=count+1        
    clear_output(wait=True)